# Unit 1 Assignment: Identifying Particles by Time of Flight
*AI/ML in Physics (CWRU)*

**Due:** end of Week 4, on Canvas. Submit this notebook (run top to bottom, all outputs showing) **and** a write-up of at most 2 pages.

**The problem:** a collider detector records thousands of charged particles. For each one you know its momentum, how far it flew, how long it took, and how much energy it deposited along the way. Each particle is a pion, a kaon or a proton, but nothing tells you which. Can unsupervised clustering sort them out?

In the lecture you saw that the choice of **features** matters as much as the choice of algorithm. This assignment is about that idea: you'll start with a clustering that fails completely and fix it with one line of physics.

### What you'll practice
1. Building a baseline and explaining why it fails.
2. Using PCA, and recognizing what it can and can't find.
3. Turning raw measurements into a physics-motivated feature.
4. Finding where a method stops working, and predicting it from the physics.
5. Arguing for the right-sized tool.

**Rules:** the true species labels are in `truth`. You may use them **only to score** a result (with `adjusted_rand_score`) or where an exercise says so. Never pass them to a clustering algorithm.

**AI use:** allowed for code, per the syllabus. Add a short note at the end saying which tools you used and for what. Your written answers must be your own reasoning.

**Packages:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`.

---
## Part 0: The detector and the data

A time-of-flight (TOF) detector measures how long a particle takes to travel from the collision point to a detector layer. For a particle of mass $m$ and momentum $p$:
$$\beta = \frac{v}{c} = \frac{p}{\sqrt{p^2 + m^2}} \qquad (\text{with } c = 1 \text{ in GeV units}), \qquad t = \frac{L}{\beta c}.$$

At the same momentum, heavier particles are slower, so they arrive later. That's the whole idea.

The simulation below is set up like a real TOF system (the ALICE detector at the LHC is similar):
- **Momentum** $p$: 0.3 to 3.0 GeV/c, measured precisely (we'll ignore its error).
- **Path length** $L$: 3.0 to 4.5 m. It varies because particles leave the collision at different angles.
- **Flight time** $t$: timing resolution 80 ps (0.080 ns).
- **Energy loss** $dE/dx$: in units of a minimum-ionizing particle. Simplified here to $dE/dx \propto 1/\beta^2$ with 8% resolution.
- **Mix:** 70% pions ($m = 0.140$ GeV/c²), 15% kaons (0.494), 15% protons (0.938).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score

C_LIGHT = 0.299792458                         # m/ns
MASSES = {"pion": 0.13957, "kaon": 0.49368, "proton": 0.93827}   # GeV/c^2
SIGMA_T = 0.080                               # ns

def simulate_tof(n=5000, seed=7):
    '''Simulated TOF tracks. Returns (tracks DataFrame, truth array); truth is 0/1/2 = pion/kaon/proton.'''
    rng = np.random.default_rng(seed)
    truth = rng.choice(3, size=n, p=[0.70, 0.15, 0.15])
    m = np.array(list(MASSES.values()))[truth]
    p = rng.uniform(0.3, 3.0, n)
    L = rng.uniform(3.0, 4.5, n)
    beta = p / np.sqrt(p**2 + m**2)
    t = L / (beta * C_LIGHT) + rng.normal(0, SIGMA_T, n)
    dedx = (1 / beta**2) * rng.normal(1, 0.08, n)
    tracks = pd.DataFrame(dict(p=p, L=L, t=t, dedx=dedx))
    return tracks, truth

tracks, truth = simulate_tof()
SPECIES = list(MASSES)
print(len(tracks), "tracks")
tracks.head()

---
## Part 1: Look, then try the obvious thing

### Exercise 1: Plot the raw data
Make two plots side by side:
1. Flight time `t` against momentum `p`.
2. **Time per meter**, `t / L`, against momentum `p`.

Use small, semi-transparent points (`s=2, alpha=0.3`). Don't color by species.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
# YOUR CODE HERE
raise NotImplementedError()
plt.tight_layout()
plt.show()

**Q1.** How many bands can you see in each plot? Why are the bands so much sharper in the second plot? Why do the bands merge at high momentum?

*Your answer here.*

### Exercise 2: The baseline
Run `KMeans(n_clusters=3, n_init=10, random_state=0)` on all four raw features (`p`, `L`, `t`, `dedx`), standardized with `StandardScaler`. Report the ARI against `truth`. Save the labels as `labels_base`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

**Q2.** The ARI should be close to 0. Look at what k-means split on: compare the mean `p` and mean `L` of each k-means group (try `tracks.groupby(labels_base).mean()`). What did k-means actually find, and why?

*Your answer here.*

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

---
## Part 2: Would dimensionality reduction help?

**Principal component analysis (PCA)** rotates the feature space so the first axis (PC1) points along the direction of greatest variance, PC2 along the next, and so on. It's often used before clustering to keep the "important" directions and drop the rest.

### Exercise 3: PCA on the raw features
1. Fit `PCA()` to the standardized raw features `X_raw`. Print the fraction of variance each component explains (`explained_variance_ratio_`) and the loadings (`components_`, one row per component, one column per feature).
2. Plot PC2 against PC1. **For this plot only**, color the points by `truth` so you can see whether PCA separated the species.
3. Run `KMeans(3)` on PC1 and PC2 only and report the ARI. Save the labels as `labels_pca`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

**Q3.** Does PCA help? PCA keeps the directions with the most variance. Is "most variance" the same as "most information about species"? Look at the last component: it explains well under 1% of the variance. What features does it combine, and why might it carry more species information than PC1? Why can't any single principal component fully separate the species?

*Your answer here.*

---
## Part 3: One line of physics

From $\beta = L/(ct)$ and $p = \gamma m \beta$, you can solve for the particle's mass using only things you measured:
$$m^2 = p^2\left(\frac{1}{\beta^2} - 1\right) = p^2\left[\left(\frac{c\,t}{L}\right)^2 - 1\right].$$

This is a **feature built from physics**. Every pion should give $m^2 \approx 0.019$, every kaon $\approx 0.244$, every proton $\approx 0.880$ (GeV/c²)², whatever its momentum or path length.

### Exercise 4: Cluster on reconstructed mass
1. Add columns `beta` and `m2` to `tracks`.
2. Histogram `m2` over the range −0.3 to 1.3 with 150 bins. Mark the true $m^2$ of each species with a vertical dotted line.
3. Fit `GaussianMixture(n_components=3, random_state=0)` to `m2` alone (it needs a 2-D array: `tracks[["m2"]]`). Report the ARI and the fitted masses, $\sqrt{\text{means}}$, sorted. Save the labels as `labels_m2`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

**Q4.** Compare this ARI with Exercises 2 and 3. The algorithm is no more complicated than k-means. What changed? Why do the fitted masses come out so close to the true values even though the algorithm never saw a label? Some tracks have $m^2 < 0$, which is unphysical. Where do they come from?

*Your answer here.*

---
## Part 4: Where does it stop working?

### Exercise 5: Separation vs. momentum
1. Split the tracks into momentum bins `[0.3, 1.0, 1.5, 2.0, 2.5, 3.0]`. In each bin, fit a fresh 3-component GMM to `m2` and report the ARI.
2. Now predict it from physics. The difference in flight time between a kaon and a pion with the same $p$ over path length $L$ is
$$\Delta t_{K\pi} = \frac{L}{c}\left(\sqrt{1 + \frac{m_K^2}{p^2}} - \sqrt{1 + \frac{m_\pi^2}{p^2}}\right).$$
Divide by the timing resolution to get the separation in standard deviations, $n_\sigma = \Delta t / \sigma_t$. Plot $n_\sigma$ against $p$ (from 0.3 to 3 GeV/c) for kaons vs. pions and for protons vs. kaons, using $L = 3.75$ m. Draw a horizontal line at $n_\sigma = 3$.

In [ ]:
edges = [0.3, 1.0, 1.5, 2.0, 2.5, 3.0]
# YOUR CODE HERE
raise NotImplementedError()

**Q5.** At what momentum does the kaon/pion separation drop below 3σ? Does that match where the ARI starts to fall? If you needed to identify kaons at 3 GeV/c, what would you change: the algorithm, or the detector? Name one specific change and say how it enters $n_\sigma$.

*Your answer here.*

### Exercise 6: Is more information always better?
At high momentum, maybe the energy loss can help. Take only tracks with $p > 2$ GeV/c.
1. Fit a 3-component GMM to `m2` alone and report the ARI.
2. Fit a 3-component GMM to `m2` **and** `dedx`, standardized together, and report the ARI.
3. Plot `dedx` against `m2` for these tracks, colored by `truth`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

**Q6.** Did adding $dE/dx$ help? Use the plot and the physics of $dE/dx \propto 1/\beta^2$ to explain why or why not. What does this say about adding features "just in case"?

*Your answer here.*

---
## Part 5: Tool-leveling (answer in the write-up)

Your write-up (2 pages max) should answer the three standard questions for this unit, then the two below:

1. **Why was clustering appropriate here?**
2. **What simpler and what more complex tools did you compare, and how did they do?** Include a small table of the ARIs from Exercises 2 through 4.
3. **Where does the method fail on this data, and why?**
4. **Simulation labels.** Real experiments train supervised classifiers (boosted trees, neural networks) on *simulated* particles where the species is known. When would that be worth the extra cost over the $m^2$ clustering? What could go wrong if the simulation doesn't match the real detector? (Hint: think about the "representative learning set" idea from Unit 0.)
5. **An LLM.** A classmate suggests giving the raw track table to an LLM and asking it to label each track. Give one reason this is a poor choice for this problem and one place in your workflow where an LLM could reasonably help.

*Your answer here.*

---
**AI use note:** *State which AI tools you used, if any, and for what.*